# 00 · What can past skeleton motion add?


We predict a teacher's future person-region feature vector from the first 32
frames of a clip. The RGB reference already sees video features, framing,
recording conditions and observation quality. The comparison asks whether an
explicit skeleton representation improves that prediction. Skeletons come from
RGB, so an improvement concerns representation for these predictors.

**Run this notebook independently in a fresh kernel.** The default
`teach` mode uses small generated examples. Set `FI_TUTORIAL_MODE=inspect`
and `FI_RUN_ROOT` before starting the kernel to read saved artifacts.
Set `FI_TUTORIAL_MODE=execute` with an explicit `FI_RUN_ROOT` to run the
production stages below. Execute notebooks **00 → 04** in order for the
full Experiment 0; each uses a fresh kernel and the same run directory.
Use the [notebook HAIC launchers](../../../slurm/future-innovation/NOTEBOOKS.md)
for scheduled execution. Inspection remains read-only. An absent local
file says nothing about the current state of a remote HAIC job.

[Study overview](../../../docs/studies/future-innovation/README.md) ·
[Historical direct-v2 specification](../../../docs/studies/future-innovation/direct-gate-protocol.md) ·
[Calibrated direct-v3 specification](../../../docs/studies/future-innovation/direct-v3-repair-protocol.md)

In [ ]:
from pathlib import Path
import os
import sys
from time import perf_counter

started = perf_counter()
override = os.environ.get("GAVD6_ROOT")
if override:
    candidates = [Path(override).expanduser().resolve()]
else:
    candidates = []
    for base in (Path.cwd(), *Path.cwd().parents):
        candidates.extend((base, base / "gavd6", base / "experiments/sjepa/gavd6"))
PROJECT_ROOT = next((p for p in candidates if (p / "src/gavd6_sjepa").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set GAVD6_ROOT to the checkout containing src/gavd6_sjepa.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from matplotlib_inline.backend_inline import set_matplotlib_formats
get_ipython().run_line_magic("matplotlib", "inline")
set_matplotlib_formats("svg", "png")
plt.rcParams.update({"figure.figsize": (8, 3), "axes.spines.top": False,
                    "axes.spines.right": False, "font.size": 11})

from gavd6_sjepa.research_directions.future_innovation.fi_tutorial_inspection import (
    artifact_inventory, inspect_report, read_optional_table, inspection_audit_path,
)

# Use "execute" for real stages or "inspect" for saved artifacts.
# Relative paths resolve from GAVD6_ROOT. Execution requires an explicit run root.
MODE = os.environ.get("FI_TUTORIAL_MODE", "teach")
if MODE not in {"teach", "inspect", "execute"}:
    raise ValueError("FI_TUTORIAL_MODE must be teach, inspect, or execute.")
if MODE == "execute" and not os.environ.get("FI_RUN_ROOT"):
    raise ValueError("Set FI_RUN_ROOT explicitly before executing real experiment stages.")
RUN_ROOT = Path(os.environ.get("FI_RUN_ROOT", "outputs/future-innovation-direct-v3-dev-20260911")).expanduser()
if not RUN_ROOT.is_absolute():
    RUN_ROOT = PROJECT_ROOT / RUN_ROOT
RUN_ROOT = RUN_ROOT.resolve()
print("Teaching examples only; no empirical gait findings." if MODE == "teach"
      else f"{MODE.upper()} mode: {RUN_ROOT}")
if MODE == "execute":
    from gavd6_sjepa.research_directions.future_innovation.fi_notebook_workflow import (
        initialize_from_environment, run_stage, build_notebook_report, finish_notebook_report,
        attempt_stage, require_stage_success,
    )
    if (RUN_ROOT / "config/run-contract.json").is_file():
        import json
        saved_run = json.loads((RUN_ROOT / "config/run-contract.json").read_text())
        print("Frozen protocol:", saved_run.get("protocol", "legacy-v1"),
              "— gate clips:", saved_run.get("cohort_size"))
        if saved_run.get("protocol", "legacy-v1") == "legacy-v1":
            print("This run retains legacy selectivity gates. Use a new run root for direct-v2.")

## Execute this stage

Initialize the immutable protocol and input provenance, or validate the existing run. Cached direct-v3 uses FI_PARENT_ROOT and a passing FI_CALIBRATION record; other protocols use the original HAIC input setup.

This cell runs only in `execute` mode. Each command uses this kernel's Python and the existing production CLI; stage logs are retained alongside the executed notebook.

In [ ]:
if MODE == "execute":
    initialize_from_environment(RUN_ROOT)

The inspected `gate-v2` / `direct-v2` run completed with STOP. Its residual head
added an unrestricted RGB map to ridge's own training errors. Unsupported weights
and unstable missingness scaling damaged prediction. The saved result stays
historical evidence; deleting weights afterward was a diagnostic intervention.

`direct-v3` fits `intercept + X W_x + S W_s` jointly. X is safely transformed RGB
and nuisance information; S contains ordered skeleton summaries. Separate ridge
penalties restrain the two coefficient blocks. The squared Frobenius penalty is
the sum of squared matrix entries. Source weights balance videos, and the
intercept is unpenalized. Inner validation can choose the exact RGB baseline.
Legacy-v1 retains its selectivity rules; direct-v2 and direct-v3 omit them.

For an invented target of 0.8, a baseline of 0.5 and full prediction of 0.7,
the improvement is visible as smaller squared error. A baseline-only outcome
returns 0.5 exactly, including after saving and reloading. The difference
`full - baseline` describes a prediction difference; direct-v3 does not train a
second RGB map on ridge residuals.

In [ ]:
if MODE == "teach":
    display(pd.DataFrame({"target": [0.8]*3, "prediction": [0.5, 0.7, 0.5]},
                         index=["RGB reference", "Illustrative joint prediction", "Exact baseline fallback"]))
    from gavd6_sjepa.research_directions.future_innovation.fi_metrics import score_arrays
    target = np.array([[-1.], [0.], [1.]])
    reference = np.zeros_like(target)
    base = target * 0.5
    fallback = score_arrays(target, base, base.copy(), np.ones(3), np.ones(1,dtype=bool))[0]
    assert fallback['delta_r2'] == 0.0
    display(pd.DataFrame([fallback]))

The main estimate is real-skeleton R² minus no-skeleton R². No-skeleton retains
time-varying joint validity while zeroing coordinates and confidence. Gain over
the shared RGB-only reference is a separate requirement. R² compares prediction
error with the outer-training-mean error; it can be negative. The target mean is
zero in training-standardized units, never the test-set mean.

The inspected cohort is development data. Passing all repaired criteria would
justify independent-source confirmation before scaling. Synthetic examples in
teach mode illustrate software behavior and cannot authorize scientific ADVANCE.

In [ ]:
if MODE != "teach":
    from gavd6_sjepa.research_directions.future_innovation.fi_contracts import read_json
    path = RUN_ROOT / 'config/run-contract.json'
    if path.is_file():
        run = read_json(path)
        display(pd.DataFrame([{'run_id':run['run_id'], 'protocol':run.get('protocol','legacy-v1'),
                               'root':str(RUN_ROOT), 'synthetic':run['synthetic'],
                               'development':run.get('development',True)}]))
    display(artifact_inventory(RUN_ROOT))
    evidence = inspect_report(RUN_ROOT)
    print('Report integrity checked:', evidence['seal_verified'])
    numerical = RUN_ROOT / 'reports/numerical-verification.json'
    print('Saved numerical reconstruction record present:', numerical.is_file())
    print('This inventory checks local presence. The report seal checks report integrity; neither is a fresh numerical reconstruction.')

## What this step establishes

The question, RGB reference and matched skeleton increment are distinct. Next establish which clips and source partitions supply the inputs and targets.

Continue with [01_cohort_and_alignment.ipynb](01_cohort_and_alignment.ipynb).

In [ ]:
print(f"Notebook elapsed time: {perf_counter() - started:.2f} seconds ({MODE} mode).")